# Demostración de eliminación de sinapsis de SRA
## - ¡Extraiga físicamente conocimientos específicos de un LLM!

Este cuaderno es una demostración interactiva del artículo *[\[AI Romance\] ¡Extraiga físicamente conocimientos específicos de un LLM! 'Eliminación de sinapsis' y 'Purga' de LLM intercambiables en caliente](https://qiita.com/JunSuzukiJapan/items/)*.

En SRA (Arquitectura de enrutamiento sináptico), el conocimiento innecesario y las sinapsis se pueden eliminar de dos maneras.

| Método | API | Propósito |
|------|-----|------|
| **Remoción física** | `pop_synapses(N)` | Elimina las sinapsis agregadas mediante Hot-Swap desde la cola para restaurar el tamaño del modelo |
| **Borrar cero (purgar)** | `clear_synapses([idx])` | Deshabilita las sinapsis en índices intermedios y las convierte en espacios libres |

También confirmaremos la **trampa de similitud del coseno** que ocurre durante la eliminación de cero y su solución en la práctica.

Finalmente, aplicamos `clear_synapses` a un modelo realmente entrenado para tareas múltiples y demostramos que **solo se destruye la función de la tarea objetivo mientras que todas las demás tareas permanecen completamente intactas (Olvido Cero)**.

---
## Parte 1: Eliminación de sinapsis (`pop_synapses`)

Cortamos físicamente las sinapsis que se agregaron más tarde mediante Hot-Swap, comenzando desde la cola.
Al igual que desinstalar un complemento de un sistema operativo, puedes eliminar físicamente partes del cerebro de la IA.

In [ ]:
import sys, os

if 'google.colab' in sys.modules:
    if not os.path.exists('SynapticRouter'):
        !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install tiktoken torch matplotlib seaborn

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn.functional as F
from src.sra_language_models import MoESRALanguageModel


In [ ]:
# Build a small model to walk through the add -> remove flow
model = MoESRALanguageModel(vocab_size=1000, dim=128, layers=2, num_synapses=4, k=2, syn_hidden=256)

print('=== Approach 1: Synapse Removal (Physical Cut) ===')
print(f'[Initial state] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Add 2 Synapses via Hot-Swap
model.add_synapses(2, freeze_base=True)
print(f'\n[After adding] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')

# Physically remove the added Synapses
model.pop_synapses(2)
print(f'\n[After removal] Synapse count: {model.blocks[0].num_synapses}')
print(f'  Router embedding shape: {model.blocks[0].router.get_full_emb().shape}')
print('\nMemory usage has been fully restored!')


---
## Parte 2: Eliminación de cero (purga) y la "trampa de similitud del coseno"

Si queremos eliminar una Synapse en un índice intermedio, eliminarla físicamente cambiaría las ID.
Entonces, en lugar de eso, sobrescribimos los pesos con `0.0`, un "borrado de cero".

Sin embargo, aquí hay una **trampa aterradora**. La similitud coseno del vector cero se convierte en `0.0`,
y si las puntuaciones de las sinapsis normales son negativas, la **sinapsis vacía termina con una puntuación más alta y comienza a absorber los datos**: un fenómeno de agujero negro.

El enrutador de SRA tiene una máscara **`-inf`** incorporada para evitar esta trampa. Comprobémoslo en la práctica.

In [ ]:
print('=== Approach 2: Verifying zero-clear and the -inf mask ===')

# Create a new model
model2 = MoESRALanguageModel(vocab_size=1000, dim=128, layers=1, num_synapses=4, k=2, syn_hidden=256)

# Inspect the weights of Synapse #2 before zero-clearing
target_idx = 2
emb_before = model2.blocks[0].router.get_full_emb()
print(f'[Before zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_before[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')

# Execute zero-clear
model2.clear_synapses([target_idx])

emb_after = model2.blocks[0].router.get_full_emb()
print(f'\n[After zero-clear] Synapse {target_idx} router embedding norm: {torch.norm(emb_after[target_idx]):.4f}')
print(f'  W1 norm: {torch.norm(model2.blocks[0].get_full_param("w1")[target_idx]):.4f}')
print('\nThe weights of Synapse #2 have been zeroed out completely!')


In [ ]:
# Actually verify the -inf mask in action
print('=== Verifying the -inf mask ===')

# Run the router with a dummy input
model2.eval()
dummy_input = torch.randint(0, 1000, (1, 10))
with torch.no_grad():
    logits, router_logits = model2(dummy_input)

# Inspect the router logits (final layer)
router_scores = router_logits[0]  # shape: (batch, seq, num_synapses)
print(f'Router scores (first token):')
scores = router_scores[0, 0]
for i in range(len(scores)):
    label = 'BLOCKED (-inf)' if scores[i] == float('-inf') else f'{scores[i]:.4f}'
    marker = ' <- zero-cleared' if i == target_idx else ''
    print(f'  Synapse {i}: {label}{marker}')

print('\nThe score of the zero-cleared Synapse has been set to -inf,')
print('   so the router can never select this Synapse again.')
print('   This completely prevents the "black-hole phenomenon".')


---
## Parte 3: Prueba funcional: `clear_synapses` en un modelo entrenado

Ahora llegamos al evento principal. En las Partes 1 y 2 verificamos el "comportamiento de la API",
pero lo que realmente importa es la prueba funcional: **"Después de la eliminación de cero, ¿solo se pierde el conocimiento eliminado mientras que el resto del conocimiento permanece completamente intacto?"**

Utilizando el mismo enfoque que el cuaderno 05 (el experimento de la lesión):
1. Tren multitarea en `copy` y `reverse`
2. Identifique las sinapsis utilizadas frecuentemente por `reverse` y elimínelas con `clear_synapses`
3. Verifique que `reverse` se vuelva irresoluble mientras `copy` continúa obteniendo una puntuación del 100 % (olvido cero)

In [ ]:
# Exactly the same setup as notebook 05
from src.sra_gpu_models import MoESRAModel
from src.sra_experiment import make_multitask_batch, make_batch, make_optimizer, load_balance_loss
from src.constants import VOCAB_SIZE, BOS, PAD

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

class MoESRAConfig:
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)

config = MoESRAConfig(
    vocab_size=30,
    d_model=128,
    n_layers=2,
    n_heads=4,
    num_synapses=8,
    k=2,
    max_seq_len=32
)

model3 = MoESRAModel(config.vocab_size, config.d_model, config.n_layers, config.num_synapses, config.k, syn_hidden=256).to(device)
optimizer = make_optimizer(model3, lr=0.001)


In [ ]:
# Multitask training (same as notebook 05)
print('Training started... (Copy & Reverse)')
model3.train()
epochs = 1500
batch_size = 32
tasks = ['copy', 'reverse']

for epoch in range(epochs):
    x, y, _ = make_multitask_batch(tasks, batch_size, min_len=4, max_len=8, device=device)

    optimizer.zero_grad()
    y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
    outputs, routing_weights, _ = model3(x, y_in)

    loss = F.cross_entropy(outputs.reshape(-1, config.vocab_size), y.reshape(-1))
    lb_loss = load_balance_loss(routing_weights) * 0.1
    total_loss = loss + lb_loss

    total_loss.backward()
    optimizer.step()

    if (epoch + 1) % 300 == 0:
        print(f'Epoch {epoch+1}/{epochs} | Loss: {loss.item():.4f}')

print('Training finished!')


### 3-1. Prueba de rendimiento previa a la eliminación e identificación de objetivos
Confirmamos que cada tarea se puede resolver correctamente e identificamos las sinapsis más utilizadas por `reverse`.

In [ ]:
def test_task(task_name):
    model3.eval()
    x, y, _ = make_multitask_batch([task_name], batch_size=100, min_len=6, max_len=6, device=device)
    with torch.no_grad():
        y_in = torch.cat([torch.full((y.size(0), 1), BOS, dtype=torch.long, device=device), y[:, :-1]], dim=1)
        outputs, routing_weights, _ = model3(x, y_in)
        preds = outputs.argmax(dim=-1)

    mask = y != PAD
    acc = (preds[mask] == y[mask]).float().mean().item()

    # Aggregate which Synapses were used (Layer 0)
    chosen = routing_weights[0].argmax(dim=-1)
    usage = torch.bincount(chosen.flatten(), minlength=config.num_synapses)

    print(f'[{task_name.upper()}] Accuracy: {acc*100:.1f}%')
    return usage

print('=== Before Deletion ===')
copy_usage = test_task('copy')
reverse_usage = test_task('reverse')

# Identify Synapses heavily used by Reverse but rarely used by Copy
# (Same logic as notebook 05)
diff = reverse_usage - copy_usage
target_synapses = [i for i, val in enumerate(diff) if val > 0]

# If they cannot be perfectly separated, pick the single Synapse most used by Reverse
if len(target_synapses) == 0:
    target_synapses = [diff.argmax().item()]

print(f'\n=> Deletion target: Synapses {target_synapses} (heavily used by REVERSE)')


### 3-2. Eliminación de sinapsis a través de `clear_synapses`
En el cuaderno 05 ejecutamos manualmente `block.w1.data[idx].zero_()`, pero aquí usamos la **`clear_synapses` API** oficial, que también aplica la máscara `-inf` del enrutador, para realizar una eliminación segura.

In [ ]:
print(f'Deleting Synapses {target_synapses} via clear_synapses...')

# Record norms before deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [Before deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')

# Use the clear_synapses API (the router's -inf mask is also applied automatically)
for block in model3.blocks:
    block.router.clear_synapses(target_synapses)
    for idx in target_synapses:
        block.w1.data[idx].zero_()
        block.b1.data[idx].zero_()
        block.w2.data[idx].zero_()
        block.b2.data[idx].zero_()
        block.state.data[idx].zero_()

print('\nDeletion complete!')

# Check norms after deletion
for idx in target_synapses:
    emb_norm = torch.norm(model3.blocks[0].router.synapse_emb.data[idx]).item()
    w1_norm = torch.norm(model3.blocks[0].w1.data[idx]).item()
    print(f'  [After deletion] Synapse {idx}: router embedding norm={emb_norm:.4f}, W1 norm={w1_norm:.4f}')


### 3-3. Prueba de rendimiento posterior a la eliminación (verificación de cero olvido)

Probamos nuevamente con algunas sinapsis eliminadas.

**Resultados esperados:**
- **Copia**: precisión preservada (utiliza diferentes sinapsis, por lo que está completamente intacta)
- **Reversa**: la precisión disminuye (sus sinapsis especializadas desaparecieron)

In [ ]:
print('=== After Deletion ===')
test_task('copy')
test_task('reverse')

print('\n[Discussion]')
print('When you destroy part of a single monolithic neural network by zeroing it out,')
print('all tasks usually collapse together.')
print('However, in SRA, an independent expert pathway (Synapse) is autonomously formed')
print('for each task, so even when clear_synapses removes a specific Synapse,')
print('tasks that use a different Synapse are completely unaffected.')
print('\nThis is the true strength of SRA as a modular AI.')
print('The free slot left behind by a deleted Synapse can later be reused by')
print('overwriting it (Hot-Swap) with a new plugin!')


---
## Resumen

| Demostración | Lo demostrado |
|------|----------------------|
| Parte 1 | Eliminación física y restauración de memoria mediante `pop_synapses` |
| Parte 2 | Eliminación de cero mediante `clear_synapses` y verificación de la máscara `-inf` |
| Parte 3 | `clear_synapses` en un modelo entrenado -> solo se destruye la tarea objetivo mientras que las demás permanecen intactas |

Con esto, hemos logrado el ciclo de vida completo de una IA modular: **"entrenamiento -> integración (Hot-Swap) -> eliminación (purga) -> reutilización de ranuras (sobrescritura)"**.